# 02 — Data Preprocessing

This notebook recreates the thesis preprocessing pipeline in a single reproducible workflow:

1. read the two German hourly electricity-load CSV files;
2. parse ENTSO-E-style German local timestamps and convert them to UTC;
3. clean the grid-load target;
4. load and validate the weather data produced by Notebook 01;
5. merge load and weather one-to-one on the UTC timestamp;
6. create calendar features using German local time;
7. create chronological train/validation/test splits.

**Important:** no global standardization or normalization is performed here. Any scaler must be fitted on the training split only inside the model pipeline to avoid data leakage. Likewise, the historical lookback sequences are created later by the forecasting notebook, not here.


In [ ]:
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
from dateutil.easter import easter


## 1. Configuration

The resolver accepts either the organized `data/...` layout below or the original files placed directly in the project root.

In [ ]:

# Resolve the repository root whether Jupyter starts in the project root
# or directly inside the notebooks/ directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

RAW_LOAD_DIR = PROJECT_ROOT / 'data' / 'raw' / 'load'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOAD_FILENAMES = [
    'Actual_consumption_201501010000_202501010000_Hour.csv',
    'Actual_consumption_202501010000_202601010000_Hour.csv',
]

WEATHER_FILENAMES = [
    'weather_germany_hourly_2015_2025.csv',  # output of Notebook 01
    'final_weather.csv',                     # compatibility with the original workflow
]

TARGET_SOURCE_COLUMN = 'grid load [MWh] Calculated resolutions'
LOCAL_TIMEZONE = 'Europe/Berlin'

TRAIN_END_YEAR = 2023
VALIDATION_YEAR = 2024
TEST_YEAR = 2025


def resolve_existing_file(filename: str, directories: list[Path]) -> Path:
    for directory in directories:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    searched = '\n'.join(str(directory / filename) for directory in directories)
    raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')


load_files = [
    resolve_existing_file(name, [RAW_LOAD_DIR, PROJECT_ROOT])
    for name in LOAD_FILENAMES
]

weather_file = None
for name in WEATHER_FILENAMES:
    try:
        weather_file = resolve_existing_file(name, [PROCESSED_DIR, PROJECT_ROOT])
        break
    except FileNotFoundError:
        pass

if weather_file is None:
    raise FileNotFoundError(
        'Weather file not found. Run 01_weather_collection_clean.ipynb first '
        'or place final_weather.csv in the project root.'
    )

print('Load files:')
for path in load_files:
    print(' -', path)
print('Weather file:', weather_file)


## 2. Read the electricity-load files

In [ ]:

load_frames = [pd.read_csv(path, sep=';', dtype=str) for path in load_files]
electricity_raw = pd.concat(load_frames, ignore_index=True)

required_load_columns = {'Start date', 'End date', TARGET_SOURCE_COLUMN}
missing_columns = required_load_columns.difference(electricity_raw.columns)
if missing_columns:
    raise ValueError(f'Missing load columns: {sorted(missing_columns)}')

print('Raw shape:', electricity_raw.shape)
print('Raw missing values in required fields:')
print(electricity_raw[list(required_load_columns)].isna().sum())
print('Duplicate local Start date strings:', electricity_raw['Start date'].duplicated().sum())



### Why duplicate local timestamps are expected

At the end of daylight-saving time, Germany observes the local hour from 02:00 to 03:00 twice. Therefore duplicate **naive local** `Start date` strings are expected. They must not be deleted. After timezone localization, the two hours receive different UTC offsets and become unique UTC timestamps.


## 3. Parse timestamps and clean the load target

In [ ]:

start_local_naive = pd.to_datetime(
    electricity_raw['Start date'],
    format='%b %d, %Y %I:%M %p',
    errors='raise',
)

# `ambiguous="infer"` uses sequence order to distinguish the two repeated
# 02:00 hours during the autumn DST transition. A nonexistent spring hour
# should not occur in the source and therefore raises an error if encountered.
electricity_raw['timestamp'] = (
    start_local_naive
    .dt.tz_localize(
        LOCAL_TIMEZONE,
        ambiguous='infer',
        nonexistent='raise',
    )
    .dt.tz_convert('UTC')
)

electricity_raw['grid_load'] = pd.to_numeric(
    electricity_raw[TARGET_SOURCE_COLUMN]
    .str.replace(',', '', regex=False)
    .str.strip(),
    errors='raise',
)

electricity = electricity_raw[['timestamp', 'grid_load']].copy()
electricity = electricity.sort_values('timestamp').reset_index(drop=True)

electricity.head()


## 4. Validate the cleaned electricity series

In [ ]:

assert electricity['timestamp'].notna().all(), 'Missing timestamps detected.'
assert electricity['grid_load'].notna().all(), 'Missing grid-load values detected.'
assert not electricity['timestamp'].duplicated().any(), 'Duplicate UTC timestamps detected.'
assert electricity['grid_load'].gt(0).all(), 'Non-positive grid-load observations detected.'
assert electricity['timestamp'].diff().dropna().eq(pd.Timedelta(hours=1)).all(), (
    'Electricity UTC timeline is not strictly hourly.'
)

print('Electricity shape:', electricity.shape)
print('UTC range:', electricity['timestamp'].min(), '→', electricity['timestamp'].max())
print('\nGrid-load summary:')
print(electricity['grid_load'].describe())


## 5. Load and validate weather data

In [ ]:

weather = pd.read_csv(weather_file)

# Compatibility with older CSVs accidentally saved with a pandas index.
unnamed_columns = [column for column in weather.columns if column.startswith('Unnamed:')]
if unnamed_columns:
    weather = weather.drop(columns=unnamed_columns)

required_weather_columns = ['timestamp', 'temperature_mean', 'humidity_mean']
missing_weather_columns = set(required_weather_columns).difference(weather.columns)
if missing_weather_columns:
    raise ValueError(f'Missing weather columns: {sorted(missing_weather_columns)}')

weather = weather[required_weather_columns].copy()
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True, errors='raise')
weather['temperature_mean'] = pd.to_numeric(weather['temperature_mean'], errors='raise')
weather['humidity_mean'] = pd.to_numeric(weather['humidity_mean'], errors='raise')
weather = weather.sort_values('timestamp').reset_index(drop=True)

assert not weather['timestamp'].duplicated().any(), 'Duplicate weather timestamps detected.'
assert weather.isna().sum().sum() == 0, 'Missing weather values detected.'
assert weather['timestamp'].diff().dropna().eq(pd.Timedelta(hours=1)).all(), (
    'Weather UTC timeline is not strictly hourly.'
)
assert weather['humidity_mean'].between(0, 100).all(), 'Humidity outside [0, 100]% detected.'
assert weather['temperature_mean'].between(-50, 60).all(), 'Implausible temperature detected.'

print('Weather shape:', weather.shape)
print('UTC range:', weather['timestamp'].min(), '→', weather['timestamp'].max())


## 6. Merge load and weather on UTC

In [ ]:

merged = pd.merge(
    electricity,
    weather,
    on='timestamp',
    how='inner',
    validate='one_to_one',
)
merged = merged.sort_values('timestamp').reset_index(drop=True)

load_rows_outside_overlap = len(electricity) - len(merged)
weather_rows_outside_overlap = len(weather) - len(merged)

print('Merged shape:', merged.shape)
print('Merged UTC range:', merged['timestamp'].min(), '→', merged['timestamp'].max())
print('Load rows outside common UTC overlap:', load_rows_outside_overlap)
print('Weather rows outside common UTC overlap:', weather_rows_outside_overlap)

assert not merged['timestamp'].duplicated().any(), 'Duplicate timestamps after merge.'
assert merged.isna().sum().sum() == 0, 'Missing values after merge.'
assert merged['timestamp'].diff().dropna().eq(pd.Timedelta(hours=1)).all(), (
    'Merged UTC timeline is not strictly hourly.'
)



The source load file is defined by **German local calendar time**, whereas the weather collection starts at `2015-01-01 00:00 UTC`. Consequently, the original thesis workflow has a one-hour boundary offset at the beginning/end of the common UTC range. The inner merge makes this explicit rather than manually shifting either source.


## 7. Calendar feature engineering in German local time

In [ ]:
def germany_nationwide_holidays(years):
    """Return nationwide German public-holiday dates for the requested years.

    State-specific holidays are deliberately excluded. 31 October 2017 is
    included because Reformation Day was a one-off nationwide holiday that year.
    """
    holiday_dates = set()

    for year in years:
        easter_sunday = easter(year)
        holiday_dates.update({
            date(year, 1, 1),                  # New Year's Day
            easter_sunday - timedelta(days=2), # Good Friday
            easter_sunday + timedelta(days=1), # Easter Monday
            date(year, 5, 1),                  # Labour Day
            easter_sunday + timedelta(days=39),# Ascension Day
            easter_sunday + timedelta(days=50),# Whit Monday
            date(year, 10, 3),                 # German Unity Day
            date(year, 12, 25),                # Christmas Day
            date(year, 12, 26),                # Second Christmas Day
        })

        if year == 2017:
            holiday_dates.add(date(2017, 10, 31))

    return holiday_dates


df = merged.copy()
df['local_time'] = df['timestamp'].dt.tz_convert(LOCAL_TIMEZONE)

df['hour'] = df['local_time'].dt.hour
df['day_of_week'] = df['local_time'].dt.dayofweek  # Monday=0, Sunday=6
df['is_weekend'] = (df['day_of_week'] >= 5).astype('int8')
df['month'] = df['local_time'].dt.month

years = range(df['local_time'].dt.year.min(), df['local_time'].dt.year.max() + 1)
german_holidays = germany_nationwide_holidays(years)
df['is_holiday'] = df['local_time'].dt.date.isin(german_holidays).astype('int8')

column_order = [
    'timestamp',
    'grid_load',
    'temperature_mean',
    'humidity_mean',
    'local_time',
    'hour',
    'day_of_week',
    'is_weekend',
    'month',
    'is_holiday',
]
df = df[column_order]

df.head()


## 8. Final dataset validation

In [ ]:

assert df.isna().sum().sum() == 0, 'Missing values in final dataset.'
assert not df['timestamp'].duplicated().any(), 'Duplicate UTC timestamps in final dataset.'
assert set(df['hour'].unique()).issubset(set(range(24)))
assert set(df['day_of_week'].unique()).issubset(set(range(7)))
assert set(df['is_weekend'].unique()).issubset({0, 1})
assert set(df['month'].unique()).issubset(set(range(1, 13)))
assert set(df['is_holiday'].unique()).issubset({0, 1})

print('Final shape:', df.shape)
print('UTC range:', df['timestamp'].min(), '→', df['timestamp'].max())
print('Local range:', df['local_time'].min(), '→', df['local_time'].max())
print('\nMissing values:')
print(df.isna().sum())
print('\nFeature values:')
print('hours:', sorted(df['hour'].unique()))
print('days of week:', sorted(df['day_of_week'].unique()))
print('months:', sorted(df['month'].unique()))
print('holiday counts:')
print(df['is_holiday'].value_counts().sort_index())


## 9. Chronological train / validation / test split

In [ ]:

train = df[df['local_time'].dt.year <= TRAIN_END_YEAR].copy().reset_index(drop=True)
validation = df[df['local_time'].dt.year == VALIDATION_YEAR].copy().reset_index(drop=True)
test = df[df['local_time'].dt.year == TEST_YEAR].copy().reset_index(drop=True)

assert len(train) > 0 and len(validation) > 0 and len(test) > 0, 'One or more splits are empty.'
assert train['timestamp'].max() < validation['timestamp'].min(), 'Train/validation overlap.'
assert validation['timestamp'].max() < test['timestamp'].min(), 'Validation/test overlap.'
assert train.isna().sum().sum() == 0
assert validation.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0

for name, split in [('Train', train), ('Validation', validation), ('Test', test)]:
    print(f'{name}: {split.shape}')
    print('  local range:', split['local_time'].min(), '→', split['local_time'].max())


## 10. Save model-ready tabular data

In [ ]:

FINAL_COMBINED_FILE = PROCESSED_DIR / 'final_combined.csv'
TRAIN_FILE = PROCESSED_DIR / 'train.csv'
VALIDATION_FILE = PROCESSED_DIR / 'validation.csv'
TEST_FILE = PROCESSED_DIR / 'test.csv'

df.to_csv(FINAL_COMBINED_FILE, index=False)
train.to_csv(TRAIN_FILE, index=False)
validation.to_csv(VALIDATION_FILE, index=False)
test.to_csv(TEST_FILE, index=False)

print('Saved:')
for path in [FINAL_COMBINED_FILE, TRAIN_FILE, VALIDATION_FILE, TEST_FILE]:
    print(' -', path)



## Outputs and modeling boundary

This notebook produces:

- `data/processed/final_combined.csv`
- `data/processed/train.csv`
- `data/processed/validation.csv`
- `data/processed/test.csv`

The downstream model notebook should parse `timestamp` as UTC and `local_time` as Europe/Berlin when needed. Model-specific transformations such as scaling, lag-window construction, and tensor formatting belong downstream and must respect the chronological split.
